In [27]:
import json
from collections import defaultdict

with open("relation_results.json") as f:
    relations = json.load(f)

with open("entity_info.json") as f:
    entity_info = json.load(f)

with open("entity_id_dict.json") as f:
    name_to_id = json.load(f)

In [28]:
id_to_types = {
    eid: data["types"]
    for eid, data in entity_info.items()
}

In [ ]:
PARENT = {
    # -------- Time --------
    "day": "Time",
    "generic_date": "date",
    "special_date": "date",
    "date": "Time",
    "month": "Time",
    "moon_phase": "Time",
    "measurement_unit": "Time",
    "measurement_method": "Time",
    "nakshatra": "Time",
    "other_time": "Time",

    # -------- Activity --------
    "festival": "Activity",
    "ritual_activity": "Activity",
    "mundane_activity": "Activity",
    "other_activity": "Activity",

    # -------- Concept --------
    "abstract_concept": "Concept",
    "attribute": "Concept",
    "state": "Concept",
    "action_process": "Concept",
    "measure_quantity": "Concept",
    "knowledge_linguistic": "Concept",
    "other_concept": "Concept",

    # -------- Geographical_Feature --------
    "landform": "Geographical_Feature",
    "water_body": "Geographical_Feature",
    "vegetation_region": "Geographical_Feature",
    "atmospheric_region": "Geographical_Feature",
    "other_geo": "Geographical_Feature",

    # -------- Living_Being --------
    "human_generic": "Living_Being",
    "human_individual": "Living_Being",
    "animal": "Living_Being",
    "plant": "Living_Being",
    "mythical_living_being": "Living_Being",
    "other_living": "Living_Being",

    # -------- Medical_Concept --------
    "disease": "Medical_Concept",
    "symptom_physical": "Medical_Concept",
    "symptom_mental": "Medical_Concept",
    "secretion_internal": "Medical_Concept",
    "secretion_external": "Medical_Concept",
    "remedy": "Medical_Concept",
    "other_medical": "Medical_Concept",

    # -------- Mythical_Entity --------
    "metaphysical_entity": "Mythical_Entity",
    "deity": "Mythical_Entity",
    "avatar": "Mythical_Entity",
    "being_class": "Mythical_Entity",
    "individual_figure": "Mythical_Entity",
    "mythical_creature_object": "Mythical_Entity",
    "other_myth": "Mythical_Entity",

    # -------- Phenomenon --------
    "celestial_phenomenon": "Phenomenon",
    "season": "Phenomenon",
    "natural_phenomenon": "Phenomenon",
    "other": "Phenomenon",
}

In [30]:
def is_ancestor(child, parent):
    while child in PARENT:
        if PARENT[child] == parent:
            return True
        child = PARENT[child]
    return False

In [31]:
type_rel_counts = defaultdict(int)

for item in relations:
    rel_data = item["relation_data"]
    
    if rel_data["relation"] == "NONE":
        continue
    
    src_name = rel_data["source_entity"]
    tgt_name = rel_data["target_entity"]
    rel = rel_data["relation"]
    
    if src_name not in name_to_id or tgt_name not in name_to_id:
        continue
    
    src_id = name_to_id[src_name]
    tgt_id = name_to_id[tgt_name]
    
    src_types = id_to_types.get(src_id, [])
    tgt_types = id_to_types.get(tgt_id, [])
    
    for t1 in src_types:
        for t2 in tgt_types:

            # skip identical types
            if t1 == t2:
                continue

            # skip parent-child relations BOTH DIRECTIONS
            if is_ancestor(t1, t2) or is_ancestor(t2, t1):
                continue

            type_rel_counts[(t1, rel, t2)] += 1

In [32]:
import math
from collections import defaultdict

total = sum(type_rel_counts.values())

# Marginals
type_freq = defaultdict(int)
rel_freq = defaultdict(int)
pair_freq = defaultdict(int)

for (t1, r, t2), c in type_rel_counts.items():
    type_freq[t1] += c
    type_freq[t2] += c
    rel_freq[r] += c
    pair_freq[(t1, t2)] += c

type_rel_scores = {}

for (t1, r, t2), c in type_rel_counts.items():

    p_triple = c / total
    p_t1 = type_freq[t1] / total
    p_t2 = type_freq[t2] / total
    p_r = rel_freq[r] / total

    # PMI with relation awareness
    denom = (p_t1 * p_r * p_t2) + 1e-12

    pmi = math.log(p_triple / denom + 1e-12)

    type_rel_scores[(t1, r, t2)] = pmi

In [33]:
from collections import defaultdict

type_pair_totals = defaultdict(int)

for (t1, r, t2), count in type_rel_counts.items():
    type_pair_totals[(t1, t2)] += count

type_rel_scores = {}

for (t1, r, t2), count in type_rel_counts.items():
    total = type_pair_totals[(t1, t2)]
    score = count / total
    type_rel_scores[(t1, r, t2)] = score

In [34]:
THRESHOLD = 0.6

strong_type_relations = [
    (t1, r, t2, score)
    for (t1, r, t2), score in type_rel_scores.items()
    if score > THRESHOLD and type_rel_counts[(t1, r, t2)] > 3
]

In [35]:
strong_type_relations

[('Phenomenon', 'determined_by', 'date', 0.6666666666666666),
 ('generic_date', 'determined_by', 'festival', 0.8),
 ('day', 'occurs_on', 'date', 1.0),
 ('day', 'occurs_on', 'generic_date', 1.0),
 ('date', 'occurs_on', 'day', 0.9787234042553191),
 ('generic_date', 'occurs_on', 'day', 1.0),
 ('Activity', 'occurs_on', 'day', 0.9117647058823529),
 ('special_date', 'occurs_on', 'day', 0.9655172413793104),
 ('festival', 'occurs_on', 'day', 1.0),
 ('ritual_activity', 'occurs_on', 'date', 0.8571428571428571),
 ('ritual_activity', 'occurs_on', 'generic_date', 1.0),
 ('Activity', 'occurs_on', 'generic_date', 0.6875),
 ('day', 'occurs_on', 'Activity', 0.625),
 ('ritual_activity', 'occurs_on', 'day', 0.9285714285714286),
 ('mundane_activity', 'occurs_on', 'day', 0.7142857142857143),
 ('action_process', 'occurs_on', 'day', 1.0),
 ('Concept', 'occurs_on', 'day', 0.8181818181818182),
 ('special_date', 'occurs_during', 'season', 1.0),
 ('special_date', 'occurs_during', 'Phenomenon', 0.7692307692307693

In [40]:
PMI_THRESHOLD = 1.0
MIN_COUNT = 5

strong_type_relations = [
    (t1, r, t2, type_rel_scores[(t1, r, t2)], type_rel_counts[(t1, r, t2)])
    for (t1, r, t2) in type_rel_scores
    if type_rel_scores[(t1, r, t2)] > PMI_THRESHOLD
    and type_rel_counts[(t1, r, t2)] >= MIN_COUNT
]

In [41]:
strong_type_relations

[]

In [42]:
sorted(type_rel_counts.items(), key=lambda x: -x[1])

[(('Activity', 'occurs_on', 'Time'), 52),
 (('date', 'occurs_on', 'day'), 46),
 (('Food', 'used_in', 'Activity'), 37),
 (('Activity', 'uses', 'Food'), 35),
 (('Activity', 'occurs_on', 'day'), 31),
 (('mundane_activity', 'uses', 'Food'), 29),
 (('special_date', 'occurs_on', 'day'), 28),
 (('Activity', 'occurs_during', 'Time'), 26),
 (('festival', 'occurs_on', 'Time'), 25),
 (('Activity', 'causes', 'Concept'), 25),
 (('Food', 'used_in', 'mundane_activity'), 22),
 (('Mythical_Entity', 'has', 'Concept'), 21),
 (('ritual_activity', 'occurs_on', 'Time'), 20),
 (('Living_Being', 'performs', 'Activity'), 20),
 (('generic_date', 'occurs_on', 'day'), 18),
 (('Activity', 'leads_to', 'Concept'), 17),
 (('human_generic', 'performs', 'Activity'), 15),
 (('Activity', 'includes', 'Food'), 15),
 (('human_generic', 'performs', 'ritual_activity'), 14),
 (('Living_Being', 'performs', 'ritual_activity'), 14),
 (('festival', 'occurs_on', 'day'), 13),
 (('ritual_activity', 'occurs_on', 'day'), 13),
 (('Time'

In [39]:
def get_relations_for_type(t):
    return [
        (r, t2, score)
        for (t1, r, t2), score in type_rel_scores.items()
        if t1 == t and score > 0.5
    ]

In [19]:
import json
from collections import defaultdict
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np

# ===================== LOAD =====================
with open("relation_results.json") as f:
    relations = json.load(f)

with open("entity_info.json") as f:
    entity_info = json.load(f)

with open("entity_id_dict.json") as f:
    name_to_id = json.load(f)

id_to_types = {
    eid: data["types"]
    for eid, data in entity_info.items()
}

# ===================== PARENT MAP =====================
PARENT = {
    "day": "Time","generic_date": "date","special_date": "date","date": "Time",
    "month": "Time","moon_phase": "Time","measurement_unit": "Time",
    "measurement_method": "Time","nakshatra": "Time","other_time": "Time",

    "festival": "Activity","ritual_activity": "Activity","mundane_activity": "Activity","other_activity": "Activity",

    "abstract_concept": "Concept","attribute": "Concept","state": "Concept",
    "action_process": "Concept","measure_quantity": "Concept","knowledge_linguistic": "Concept","other_concept": "Concept",

    "landform": "Geographical_Feature","water_body": "Geographical_Feature",
    "vegetation_region": "Geographical_Feature","atmospheric_region": "Geographical_Feature","other_geo": "Geographical_Feature",

    "human_generic": "Living_Being","human_individual": "Living_Being",
    "animal": "Living_Being","plant": "Living_Being","mythical_living_being": "Living_Being","other_living": "Living_Being",

    "disease": "Medical_Concept","symptom_physical": "Medical_Concept",
    "symptom_mental": "Medical_Concept","secretion_internal": "Medical_Concept",
    "secretion_external": "Medical_Concept","remedy": "Medical_Concept","other_medical": "Medical_Concept",

    "metaphysical_entity": "Mythical_Entity","deity": "Mythical_Entity",
    "avatar": "Mythical_Entity","being_class": "Mythical_Entity",
    "individual_figure": "Mythical_Entity","mythical_creature_object": "Mythical_Entity","other_myth": "Mythical_Entity",

    "celestial_phenomenon": "Phenomenon","season": "Phenomenon",
    "natural_phenomenon": "Phenomenon","other_phenomenon": "Phenomenon",
}

# ===================== FIND LEAF TYPES =====================
# leaf = types that are NOT parent of anything
all_children = set(PARENT.keys())
all_parents = set(PARENT.values())

LEAF_TYPES = all_children - all_parents

def get_leaf_types(types):
    return [t for t in types if t in LEAF_TYPES]


# ===================== UNIQUE TRIPLES =====================
unique_triples = set()

for item in relations:
    rel_data = item["relation_data"]
    if rel_data["relation"] == "NONE":
        continue
    
    unique_triples.add((
        rel_data["source_entity"],
        rel_data["relation"],
        rel_data["target_entity"]
    ))

# ===================== BUILD TYPE-LEVEL DATA =====================
sentences = []
mapping = []   # index -> (t1, relation, t2)

for (src, rel, tgt) in unique_triples:

    if src not in name_to_id or tgt not in name_to_id:
        continue

    src_types = get_leaf_types(id_to_types.get(name_to_id[src], []))
    tgt_types = get_leaf_types(id_to_types.get(name_to_id[tgt], []))

    for t1 in src_types:
        for t2 in tgt_types:

            # build sentence
            text = f"this {' '.join(t1.split('_'))} {' '.join(rel.split('_'))} this {' '.join(t2.split('_'))}"

            sentences.append(text)
            mapping.append((t1, rel, t2))


print(f"Total sentences: {len(sentences)}")

# ===================== LOAD BGE MODEL =====================
model_name = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ===================== EMBEDDING FUNCTION =====================
def get_embeddings(sentences, batch_size=32):
    all_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
            emb = outputs.last_hidden_state[:, 0]
            emb = F.normalize(emb, p=2, dim=1)
            all_embeddings.append(emb.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

# ===================== GENERATE EMBEDDINGS =====================
embeddings = get_embeddings(sentences)

print("Embedding shape:", embeddings.shape)

# ===================== SAVE =====================
np.save("type_relation_embeddings.npy", embeddings)

with open("type_relation_mapping.json", "w") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print("Saved embeddings + mapping")

Total sentences: 1428


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding shape: (1428, 384)
Saved embeddings + mapping


In [20]:
import umap
import hdbscan

# ===================== UMAP =====================
umap_model = umap.UMAP(
    n_components=15,       # your choice
    n_neighbors=15,        # controls local vs global
    min_dist=0.0,          # tighter clusters
    metric='cosine',       # important for embeddings
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

print("Reduced shape:", reduced_embeddings.shape)


# ===================== HDBSCAN =====================
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=15,
    metric='euclidean'
)

labels = clusterer.fit_predict(reduced_embeddings)

print("Clusters:", set(labels))
print("Noise points:", sum(labels == -1))

/data/shubham/miniconda/envs/namo/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced shape: (1428, 15)
Clusters: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, -1}
Noise points: 124


In [21]:
from collections import defaultdict

clusters = defaultdict(list)

for i, label in enumerate(labels):
    clusters[label].append(i)

for label, indices in clusters.items():
    print(f"\nCluster {label} (size={len(indices)})")
    
    for idx in indices[:10]:  # preview
        print(mapping[idx])


Cluster 0 (size=33)
('knowledge_linguistic', 'explains', 'ritual_activity')
('ritual_activity', 'granted_to', 'knowledge_linguistic')
('action_process', 'caused_by', 'knowledge_linguistic')
('ritual_activity', 'described_in', 'knowledge_linguistic')
('knowledge_linguistic', 'describes', 'ritual_activity')
('knowledge_linguistic', 'causes', 'action_process')
('ritual_activity', 'has', 'knowledge_linguistic')
('knowledge_linguistic', 'caused_by', 'ritual_activity')
('ritual_activity', 'leads_to', 'knowledge_linguistic')
('knowledge_linguistic', 'includes', 'ritual_activity')

Cluster 5 (size=65)
('abstract_concept', 'created_at', 'celestial_phenomenon')
('festival', 'occurs_on', 'moon_phase')
('human_generic', 'fails_to_achieve', 'moon_phase')
('moon_phase', 'is_an', 'mythical_living_being')
('moon_phase', 'is_an', 'avatar')
('moon_phase', 'is_an', 'knowledge_linguistic')
('month', 'contains', 'moon_phase')
('special_date', 'determined_by', 'celestial_phenomenon')
('celestial_phenomenon

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import numpy as np

# group indices by cluster
clusters = defaultdict(list)
for i, label in enumerate(labels):
    if label != -1:
        clusters[label].append(i)

# refine clusters using similarity threshold
SIM_THRESHOLD = 0.965

refined_clusters = []

for label, idxs in clusters.items():
    emb = embeddings[idxs]   # use ORIGINAL embeddings (important)

    sim_matrix = cosine_similarity(emb)

    used = set()
    
    for i in range(len(idxs)):
        if i in used:
            continue
        
        group = [idxs[i]]
        used.add(i)
        
        for j in range(i + 1, len(idxs)):
            if j in used:
                continue
            
            if sim_matrix[i, j] >= SIM_THRESHOLD:
                group.append(idxs[j])
                used.add(j)
        
        refined_clusters.append(group)

In [18]:
verbose_refined_clusters = [[mapping[idx] for idx in cluster] for cluster in refined_clusters]
print(json.dumps(verbose_refined_clusters[:100], indent=1, ensure_ascii=False))

[
 [
  [
   "knowledge_linguistic",
   "explains",
   "festival"
  ],
  [
   "festival",
   "related_to",
   "knowledge_linguistic"
  ],
  [
   "festival",
   "mentioned_in",
   "knowledge_linguistic"
  ],
  [
   "festival",
   "associated_with",
   "knowledge_linguistic"
  ],
  [
   "festival",
   "leads_to",
   "knowledge_linguistic"
  ],
  [
   "festival",
   "has_explanation",
   "knowledge_linguistic"
  ],
  [
   "knowledge_linguistic",
   "described_in_context_of",
   "festival"
  ],
  [
   "festival",
   "discusses",
   "knowledge_linguistic"
  ],
  [
   "knowledge_linguistic",
   "occurs_in",
   "festival"
  ],
  [
   "knowledge_linguistic",
   "occurs_during",
   "festival"
  ],
  [
   "festival",
   "has",
   "knowledge_linguistic"
  ],
  [
   "knowledge_linguistic",
   "described_in",
   "festival"
  ]
 ],
 [
  [
   "knowledge_linguistic",
   "has_purpose",
   "abstract_concept"
  ]
 ],
 [
  [
   "knowledge_linguistic",
   "performs_for",
   "mundane_activity"
  ],
  [
   "k